In [1]:
from langchain.agents import Tool, AgentType, initialize_agent
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.tools import StructuredTool
import os
from dotenv import load_dotenv

In [2]:
#Load API key
load_dotenv(dotenv_path=".env")
openai_api_key=os.getenv("OPENAI_API_KEY")

#Initialize LLM
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20372\3725230459.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [3]:
#Create Vector DB (Retriever)
with open("sample.txt","r", encoding="utf-8")as f:
    text_data= f.read()

#Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter
splitter= CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts= splitter.split_text(text_data)
embedding= OpenAIEmbeddings()
vectorstore= FAISS.from_texts(texts, embedding)
retriever= vectorstore.as_retriever()   

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20372\2236871248.py:9: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding= OpenAIEmbeddings()


In [6]:
#Conversational RAG chain
rag_chain= ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

In [9]:
#Memory for chat history
memory= ConversationBufferMemory(memory_key="chat_history", return_messages=True)

#Wrap RAG as a StructuredTool

def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history":[]
    })["answer"]

rag_tool=StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# Create agent

agent= initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    handle_parsing_errors= True
)

In [10]:
#Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)


1️⃣ First question


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: What is LangChain?
```
Observation: LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports various features such as retrieval-augmented generation (RAG), agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document question and answer systems, and AI workflows.
Thought:Do I need to use a tool? No  
AI: LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports various features such as retrieval-augmented generation (RAG), agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document question and answer systems, and AI workflows.

> Finished chain.
Answer: LangChain is a framework for building applications with large language models (LL